In [ ]:
from pathlib import Path

from datasets import load_from_disk, concatenate_datasets

# resolve the project root no matter where the kernel was started from
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
SHARD_DIR = PROJECT_ROOT / "data" / "wildchat-annotations" / "shards"

if not SHARD_DIR.is_dir():
    raise FileNotFoundError(f"no shard dir at {SHARD_DIR} - run annotate.ipynb first")

# a finished shard is a directory with state.json that is not a partial .tmp write
shard_dirs = [d for d in sorted(SHARD_DIR.iterdir())
              if d.is_dir() and not d.name.endswith(".tmp") and (d / "state.json").exists()]
if not shard_dirs:
    raise RuntimeError(f"{SHARD_DIR} exists but holds no finished shards yet - run annotate.ipynb first")

print(f"{len(shard_dirs)} finished shards under {SHARD_DIR.relative_to(PROJECT_ROOT)}")

In [ ]:
# load every shard, guard against schema drift, merge into one dataset
# Shards are stored as a dir. This is loading each arrow dataset from the completed dir
shards = [load_from_disk(str(d)) for d in shard_dirs]
annotated = concatenate_datasets(shards)
errored = annotated.filter(lambda e: e != "", input_columns="error").num_rows

print(f"{annotated.num_rows} messages from {len(shards)} shards "
    f"({errored} rows carry at least one failed request)")
annotated

In [ ]:
FIXED_COLUMNS = ["conversation_id", "message_idx", "role", "content_hash",
                 "content", "model", "rationale", "error", "timestamp"]
THEMES = [c for c in annotated.column_names if c not in FIXED_COLUMNS]

clean = annotated.filter(lambda err: err == "", input_columns="error")

# This is a way to see avg scores per theme to get a sense of signal
by_role = clean.to_pandas().groupby("role")[THEMES].mean()
by_role